# FedCore example: сжатие ResNet-152 методом pruning

Этот ноутбук демонстрирует полный пользовательский сценарий работы с моделью ResNet-152:

1. подготовка CIFAR данных;
2. создание модели ResNet-152;
3. измерение baseline-метрик;
4. применение pruning;
5. короткое дообучение после pruning;
6. сравнение качества,  эффективного размера и задержки инференса;
7. сохранение результатов.

## 1. Импорт библиотек и проверка окружения

In [1]:
import sys
import os
import torch

from pathlib import Path 

REPO_ROOT = Path(os.getcwd()).resolve().parents[2]
sys.path.insert(0, str(REPO_ROOT))

from fedcore.api.config_factory import ConfigFactory
from fedcore.api.api_configs import (APIConfigTemplate, AutoMLConfigTemplate, FedotConfigTemplate,
                                     LearningConfigTemplate, ModelArchitectureConfigTemplate,
                                     TrainingTemplate, PruningTemplate)
from fedcore.architecture.dataset.api_loader import ApiLoader
from fedcore.data.dataloader import load_data
from fedcore.tools.example_utils import get_scenario_for_api
from fedcore.api.main import FedCore
from torchvision import models

/home/leostre/.local/share/virtualenvs/FedCore-wPuHcacz/lib/python3.10/site-packages/torch/cuda/__init__.py:63: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/home/leostre/.local/share/virtualenvs/FedCore-wPuHcacz/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/leostre/.local/share/virtualenvs/FedCore-wPuHcacz/lib/python3.10/site-packages/hyperopt/atpe.py:19: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.


@@@ before bt
@@@ before op
@@@ before dlh
⚙️  Running in WANDB offline mode
2026-08-07 17:34:01,868 - Trying to use device <cuda>
2026-08-07 17:34:01,868 - Device <cuda:0> is selected
[Warning] TransMLA Core: Some functions may not be available due to missing dependencies: No module named 'utils'


## 2. Подготовка модели и данных

In [2]:

METRIC_TO_OPTIMISE = ['MulticlassAccuracy__10', 'Latency', 
                       'ModelSize']
LOSS = 'cross_entropy'
PROBLEM = 'classification'

pretrained_resnet152 = models.resnet152(weights=models.ResNet152_Weights.DEFAULT)
pretrained_resnet152.fc = torch.nn.Linear(2048, 10) 
INITIAL_ASSUMPTION = pretrained_resnet152 

train_dataloader_params = {"batch_size": 64,
                           'shuffle': True,
                           'is_train': True,
                           'data_type': 'table',
                           'split_ratio': [0.8, 0.2]}
test_dataloader_params = {"batch_size": 100,
                          'shuffle': True,
                          'is_train': False,
                          'data_type': 'table'}

def load_benchmark_dataset(dataset_name, train_dataloader_params, test_dataloader_params):
    fedcore_train_data = load_data(source=dataset_name, loader_params=train_dataloader_params)
    fedcore_test_data = load_data(source=dataset_name, loader_params=test_dataloader_params)
    return fedcore_train_data, fedcore_test_data


In [3]:
fedcore_train_data, fedcore_test_data = load_benchmark_dataset('CIFAR10', train_dataloader_params,
                                                                   test_dataloader_params)

## 3. Настройка автоматического прунинга

In [4]:
fedot_config = FedotConfigTemplate(problem='classification',
                                   metric=METRIC_TO_OPTIMISE,
                                   pop_size=1,
                                   timeout=1,
                                   initial_assumption=INITIAL_ASSUMPTION)

automl_config = AutoMLConfigTemplate(fedot_config=fedot_config)

finetune_config = TrainingTemplate(epochs=3,
                                            log_each=3,
                                            eval_each=3,
                                            )


peft_config = PruningTemplate(importance="magnitude",
                              prune_each=5,
                              epochs=9,
                              save_each=0,
                              eval_each=5,
                              pruning_ratio=0.4,
                              )

learning_config = LearningConfigTemplate(criterion='cross_entropy',
                                         learning_strategy='from_checkpoint',
                                         peft_strategy_params=[peft_config])

api_template = APIConfigTemplate(automl_config=automl_config,
                                 learning_config=learning_config)

APIConfig = ConfigFactory.from_template(api_template)
api_config = APIConfig()
    

## 4. Создание FedCore-агента и сжатие сети

In [ ]:

fedcore_compressor = FedCore(api_config)
    
fedcore_compressor.fit(fedcore_train_data)


Creating Dask Server
2026-08-07 17:34:03,580 - To route to workers diagnostics web server please install jupyter-server-proxy: python -m pip install jupyter-server-proxy
2026-08-07 17:34:03,595 - State start
2026-08-07 17:34:03,606 -   Scheduler at: inproc://10.146.216.12/56979/1
2026-08-07 17:34:03,607 -   dashboard at:  http://10.146.216.12:8787/status
2026-08-07 17:34:03,610 -       Start worker at: inproc://10.146.216.12/56979/4
2026-08-07 17:34:03,611 -          Listening to:        inproc10.146.216.12
2026-08-07 17:34:03,611 -           Worker name:                          0
2026-08-07 17:34:03,611 -          dashboard at:        10.146.216.12:37389
2026-08-07 17:34:03,611 - Waiting to connect to: inproc://10.146.216.12/56979/1
2026-08-07 17:34:03,611 - -------------------------------------------------
2026-08-07 17:34:03,612 -               Threads:                          4
2026-08-07 17:34:03,612 -                Memory:                   4.45 GiB
2026-08-07 17:34:03,612 -  

Generations:   0%|          | 0/10000 [00:00<?, ?gen/s]

Entering: _map_importance_name
  Args: ()
  Kwargs: {}
Exiting: _map_importance_name -> <torch_pruning.pruner.importance.MagnitudeImportance object at 0x76b060b9dc90>

Entering: fit
  Args: (CompressionInputData(idx=array([0]), task=Task(task_type=<TaskTypesEnum.classification: 'classification'>, task_params=None), data_type=<DataTypesEnum.image: 'image'>, features=None, categorical_features=None, categorical_idx=None, numerical_idx=array([0]), encoded_idx=None, features_names=None, target=None, supplementary_data=SupplementaryData(is_main_target=True, data_flow_length=0, features_mask=None, previous_operations=None, obligatorily_preprocessed=True, optionally_preprocessed=False, non_int_idx=None, col_type_ids=None, is_auto_preprocessed=True), train_dataloader=<torch.utils.data.dataloader.DataLoader object at 0x76b060edb100>, val_dataloader=<torch.utils.data.dataloader.DataLoader object at 0x76b060edb130>, test_dataloader=None, input_dim=32, num_classes=10, model=ResNet(
  (conv1): Conv

Batch #: 100%|██████████| 625/625 [00:29<00:00, 21.28it/s]


Entering: trigger
  Args: (1, {'val_loader': <torch.utils.data.dataloader.DataLoader object at 0x76b060edb130>, 'criterion': functools.partial(<bound method BaseTrainer._compute_loss of BaseNeuralModel
Training Scheme:
Epoch start:
	OptimizerGen
<<<Training>>>
Epoch end
	ZeroShotPruner
	FitReport
	Evaluator
	Saver>, criterion=CrossEntropyLoss()), 'history': {'train_loss': [(1, 0.98602898311615)], 'val_loss': []}})
  Kwargs: {}
Entering: is_epoch_arrived_default
  Args: (1, 5)
  Kwargs: {}
Exiting: is_epoch_arrived_default -> False

Exiting: trigger -> False



In [ ]:
model_comparison = fedcore_compressor.get_report(fedcore_test_data)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Data from the table
metrics = model_comparison.index.tolist()
original = model_comparison.loc[:, (0, 'original')].tolist()
fedcore = model_comparison.loc[:, (0, 'fedcore')].tolist()
change =  model_comparison.loc[:, (0, 'change')].tolist() # percentage change

# Colors for bars
colors_original = ['#4A90D9', '#4A90D9', '#4A90D9']
colors_fedcore = ['#2ECC71', '#2ECC71', '#2ECC71']

# Setup the plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# --- Plot 1: Side-by-side bar chart ---
x = np.arange(len(metrics))
width = 0.35

bars1 = ax1.bar(x - width/2, original, width, label='Original', color='#4A90D9', edgecolor='white', linewidth=1)
bars2 = ax1.bar(x + width/2, fedcore, width, label='FedCore', color='#2ECC71', edgecolor='white', linewidth=1)

# Add value labels on bars
for bar, val in zip(bars1, original):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, 
             f'{val:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold', color='#4A90D9')

for bar, val in zip(bars2, fedcore):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, 
             f'{val:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold', color='#2ECC71')

ax1.set_xlabel('Metrics', fontsize=13, fontweight='bold')
ax1.set_ylabel('Score / Value', fontsize=13, fontweight='bold')
ax1.set_title('Original vs FedCore Performance Comparison', fontsize=14, fontweight='bold', pad=15)
ax1.set_xticks(x)
ax1.set_xticklabels(metrics, fontsize=11)
ax1.legend(loc='upper right', fontsize=11)
ax1.set_ylim(0, 1.1)
ax1.grid(axis='y', linestyle='--', alpha=0.7)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

# --- Plot 2: Percentage Change (improvement/decline) ---
colors_change = ['#2ECC71' if c >= 0 else '#2ECC71' for c in change]
bars_change = ax2.bar(metrics, change, color=colors_change, edgecolor='white', linewidth=1.5)

# Add value labels with +/- sign
for bar, val in zip(bars_change, change):
    sign = '+' if val >= 0 else ''
    color = '#2ECC71' if val >= 0 else '#2ECC71'
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + (3 if val >= 0 else -3), 
             f'{sign}{val:.2f}%', ha='center', va='bottom' if val >= 0 else 'top', 
             fontsize=12, fontweight='bold', color=color)

# Add horizontal line at 0
ax2.axhline(y=0, color='black', linestyle='-', linewidth=1, alpha=0.5)

ax2.set_xlabel('Metrics', fontsize=13, fontweight='bold')
ax2.set_ylabel('Percentage Change (%)', fontsize=13, fontweight='bold')
ax2.set_title('FedCore Performance Improvement / Decline', fontsize=14, fontweight='bold', pad=15)
ax2.grid(axis='y', linestyle='--', alpha=0.5)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

# Adjust layout
plt.tight_layout()
plt.savefig('fedcore_comparison.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()